In [1]:
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(future))
suppressMessages(library(parallel))
suppressMessages(library(RColorBrewer))
suppressMessages(library(viridis))
suppressMessages(library(qs))
suppressMessages(library(harmony))
suppressMessages(library(igraph))
suppressMessages(library(ggraph))
suppressMessages(library(ggrepel)) 
suppressMessages(library(tidyr))
suppressMessages(library(foreach))
suppressMessages(library(doParallel))
suppressMessages(library(tidygraph))
suppressMessages(library(anndata))
suppressMessages(library(randomForest))
suppressMessages(library(pheatmap))
suppressMessages(library(dbscan))
suppressMessages(library(scDblFinder))
suppressMessages(library(cowplot))

# Python 环境
suppressMessages(library(reticulate))
RETICULATE_PYTHON <- "/home/chaiqw/project/mb/.venv/bin/python"
use_virtualenv("/home/chaiqw/project/mb/.venv")

# 资源设定
plan("multicore", workers = 64)
options(future.globals.maxSize = 100000 * 1024^5)


In [5]:
DA = readRDS('/mnt/90-connectome/Personal/CQW/midbrain/area_h5ad/20260728_VTA_DA_seurat_cluster.rds')
nonneuron = readRDS('/mnt/90-connectome/Personal/CQW/midbrain/area_h5ad/20260729_VTA_nonneuron_cluster.rds')
neuron = readRDS('/mnt/90-connectome/Personal/CQW/midbrain/area_h5ad/20260730_VTA_neuron.rds')

In [4]:
seurat = readRDS('/mnt/90-connectome/Personal/CQW/midbrain/area_h5ad/20260728_VTA_SCTseurat_harmony_rf_clean.rds')

In [6]:
merge_data <- merge(
    x = DA,
    y = list(nonneuron, neuron),
    add.cell.ids = c("DA", "nonneuron", "neuron"),
    project = "VTA_merged"
)

# 供后续 SCTransform 分析使用
seurat_1 <- merge_data

In [7]:
SCTseurat = SCTransform(
    seurat_1, assay="RNA",
    ncells=ncol(seurat_1[["RNA"]]),
    variable.features.n=3000,
    vars.to.regress="percent.mt",
    return.only.var.genes = T,
    method="glmGamPoi"
)

SCTseurat = SCTseurat %>%
    RunPCA(npcs=100, verbose = FALSE) %>%
    FindNeighbors(dims = 1:30) %>%
    FindClusters(verbose = FALSE, resolution = 0.5, random.seed = 111) %>%
    RunUMAP(dims = 1:30,umap.method = "umap-learn"
)

# 去批次
SCTseurat_harmony = SCTseurat %>%
    RunHarmony(group.by.vars = c("Animal"),dims.use = 1:30 ,theta = 2,reduction = "pca", max.iter.harmony = 10,sigma = 0.1,assay.use = "SCT", reduction.save = "harmony") %>%
    # FindNeighbors(dims = 1:30, reduction = "harmony") %>%
    # FindClusters(verbose = FALSE, resolution =0.5, random.seed = 1234) %>%
    RunUMAP(dims = 1:30, reduction = "harmony", assay = "SCT"
)

Running SCTransform on assay: RNA

vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.

Calculating cell attributes from input UMI matrix: log_umi

Variance stabilizing transformation of count matrix of size 17569 by 3630

Model formula is y ~ log_umi

Get Negative Binomial regression parameters per gene

Using 2000 genes, 3630 cells

Found 47 outliers - those will be ignored in fitting/regularization step


Second step: Get residuals using fitted parameters for 17569 genes

Computing corrected count matrix for 17569 genes

Calculating gene attributes

Wall clock passed: Time difference of 1.952249 mins

Determine variable features

Regressing out percent.mt

Centering data matrix

Place corrected count matrix in counts slot

vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.

Calculating cell attributes from input UMI matrix: log_umi

Variance stabilizing transformation of count matrix of size 18653 by 72348

Model formula is y ~ 